<a href="https://colab.research.google.com/github/JSJeong-me/GPT-Agent/blob/main/06%20Deepagent/12_MRO_Hybrid_RAG_Part4_Report_Standalone_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 4 단독 실습 — MRO Hybrid RAG 분석 결과 보고서 자동 생성

**실습 목적:** 원본 MRO `.xls`에서 정형 통계를 추출하고 Hybrid RAG 검색 결과와 결합하여 **Word 보고서, Excel 근거자료, JSON 감사기록, ZIP 패키지**를 생성합니다.

이 파일은 이전 통합 노트북의 **Part 4 (C-1~C-8)를 원형대로 분리**한 뒤, Colab 새 런타임에서도 실행되도록 데이터 준비 및 Custom Tool 선행 셀을 최소 구성으로 포함했습니다. 원본 Part 4의 기능과 보고서 생성 코드는 유지합니다.

- 입력: `2026년_3분기_DO본부_MRO팀_접수일자기준-요구사항정의서(07월)_LineUs.xls` (실제 업로드 파일명에 따라 자동 선택)
- 기본 모드: **외부 AI 호출 없음**, BM25 기반 검색 + 결정적 통계
- 선택 모드: 사내 데이터 전송 승인을 받은 경우에 한하여 `ENABLE_EXTERNAL_AI=True` 및 선택적 AI 해설
- 주의: 헤더 자동 탐지 및 컬럼 매핑 결과를 검토한 다음 실행하세요. 보고서에는 회사 정보가 남아 있을 수 있습니다.


## 0. 실행 환경 및 필수 패키지 설치

In [1]:
%pip -q install "xlrd>=2.0.1" pandas "langchain-core" "langchain-community" "langchain-openai" "langchain-chroma" "rank_bm25" "python-docx" "openpyxl"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
holidays 0.108 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
bigframes 2.51.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
tweepy 4.17.0 requires oauthli

### 0-1. 실행 설정

In [12]:
import os, re, json, hashlib, getpass, unicodedata
from pathlib import Path
import pandas as pd
from IPython.display import display

ENABLE_EXTERNAL_AI = True  # 사내 외부전송 승인을 받은 경우에만 True로 변경
CHAT_MODEL = "gpt-4o-mini"
EMBED_MODEL = "text-embedding-3-small"
WORKDIR = Path("/content/mro_hybrid_rag") if Path("/content").exists() else Path.cwd() / "mro_hybrid_rag"
WORKDIR.mkdir(parents=True, exist_ok=True)

if ENABLE_EXTERNAL_AI and not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")
print("외부 AI 사용:", ENABLE_EXTERNAL_AI, "| 작업 폴더:", WORKDIR)

OpenAI API Key: ··········
외부 AI 사용: True | 작업 폴더: /content/mro_hybrid_rag


### 0-2. 원본 MRO Excel 업로드

In [3]:
EXCEL_PATH = None  # 로컬 예: "/path/to/요구사항정의서.xls"
if EXCEL_PATH is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        assert uploaded, "업로드된 파일이 없습니다."
        EXCEL_PATH = next(iter(uploaded.keys()))
    except ImportError:
        matches = sorted(Path.cwd().glob("*.xls"))
        if not matches:
            raise FileNotFoundError("작업 디렉터리에 .xls가 없습니다. EXCEL_PATH를 지정하세요.")
        EXCEL_PATH = str(matches[0])
EXCEL_PATH = Path(EXCEL_PATH)
assert EXCEL_PATH.exists() and EXCEL_PATH.suffix.lower() == ".xls"
print("선택 파일:", EXCEL_PATH.name)

Saving 2026년_3분기_DO본부_MRO팀_접수일자기준-요구사항정의서(07월)_LineUs.xls to 2026년_3분기_DO본부_MRO팀_접수일자기준-요구사항정의서(07월)_LineUs.xls
선택 파일: 2026년_3분기_DO본부_MRO팀_접수일자기준-요구사항정의서(07월)_LineUs.xls


### 0-3. 시트 구조 확인

In [4]:
excel = pd.ExcelFile(EXCEL_PATH, engine="xlrd")
print("시트 목록:", excel.sheet_names)
SHEET_NAME = "AS" if "AS" in excel.sheet_names else excel.sheet_names[0]
raw = pd.read_excel(EXCEL_PATH, sheet_name=SHEET_NAME, header=None, dtype=object, engine="xlrd").fillna("")
print("선택 시트:", SHEET_NAME, "| 원본 크기:", raw.shape)
for idx in range(min(15, len(raw))):
    vals = [f"{j}:{str(v)[:35]}" for j,v in enumerate(raw.iloc[idx]) if str(v).strip()][:15]
    print(f"행 {idx:02d}:", " / ".join(vals))

시트 목록: ['AS관리']
선택 시트: AS관리 | 원본 크기: (32, 56)
행 00: 0:접수정보 / 5:고객사정보 / 9:문의 유형 정보 / 16:처리상태 정보 / 22:처리완료 정보 / 34:처리완료 상세 정보 / 45:팀장승인 / 48:배포승인 / 51:배포정보 / 54:배포승인 정보
행 01: 0:No / 1:접수번호 / 2:하위작업 / 3:접수일 / 4:접수경로 / 5:거래처구분 / 6:거래처코드 / 7:거래처명 / 8:의뢰자명 / 9:처리상태 / 10:부서명 / 11:처리담당자 / 12:중요도 / 13:모듈 / 14:중분류
행 02: 0:30 / 1:20260731001 / 3:20260731 / 4:방문 / 5:본사 / 6:00000 / 7:(주)중외정보기술 / 8:김경아 / 9:배포승인 / 10:MRO팀 / 11:강롱기 / 12:일반건 / 13:ERP-인사 / 16:20260812 / 17:ERP
행 03: 0:29 / 1:20260729001 / 3:20260729 / 4:방문 / 5:본사 / 6:00000 / 7:(주)중외정보기술 / 8:이준호 / 9:처리완료 / 10:MRO팀 / 11:강롱기 / 12:일반건 / 16:20260729 / 17:ERP / 18:일반
행 04: 0:28 / 1:20260728004 / 3:20260728 / 4:CALL / 5:대외기업 / 6:11654 / 7:(주)한국티이아이 / 8:최정민 / 9:처리완료 / 10:MRO팀 / 11:최정민 / 12:일반건 / 16:20260729 / 17:기타 / 18:일반
행 05: 0:27 / 1:20260728003 / 3:20260728 / 4:기타 / 5:본사 / 6:00000 / 7:(주)중외정보기술 / 8:김경아 / 9:처리완료 / 10:MRO팀 / 11:김규민 / 12:일반건 / 16:20260724 / 17:A/S서비스플랫폼 / 18:일반
행 06: 0:26 / 1:20260727006 / 3:20260727 / 4:메신저 / 5:본사 / 6:00000 

### 0-4. 헤더 탐지 및 데이터프레임 생성

In [5]:
KEYWORDS = ["접수", "번호", "고객", "문의", "처리", "승인", "배포", "작업", "요청", "상태", "담당", "조치", "완료", "등록"]
def norm_header(v):
    return re.sub(r"\s+", " ", str(v).replace("\n", " ").strip())

def header_score(i):
    vals = [norm_header(x) for x in raw.iloc[i]]
    return sum(any(k in v for k in KEYWORDS) for v in vals if v)
scores = sorted(((header_score(i), i) for i in range(min(25, len(raw)))), reverse=True)
print("헤더 후보 (점수, 0부터 시작하는 행):", scores[:8])
HEADER_ROW = scores[0][1]  # 출력 확인 후 직접 수정할 수 있음
HEADER_DEPTH = 1            # 두 행에 걸친 헤더는 2로 설정

def build_frame(raw, header_row, header_depth):
    hdr = raw.iloc[header_row:header_row+header_depth].copy().replace("", pd.NA).ffill(axis=1).fillna("")
    names = []
    for col in range(raw.shape[1]):
        parts = [norm_header(x) for x in hdr.iloc[:,col].tolist()]
        parts = list(dict.fromkeys(p for p in parts if p))
        names.append(" / ".join(parts) or f"미지정_{col}")
    count = {}; unique = []
    for name in names:
        count[name] = count.get(name, 0)+1
        unique.append(name if count[name] == 1 else f"{name}__{count[name]}")
    out = raw.iloc[header_row+header_depth:].copy()
    out.columns = unique
    out.insert(0, "_excel_row", out.index + 1)  # 실제 Excel 행 번호(1-based)
    useful = [c for c in out.columns if c == "_excel_row" or not c.startswith("미지정_") or out[c].astype(str).str.strip().ne("").any()]
    out = out[useful]
    out = out.loc[out.drop(columns="_excel_row").astype(str).apply(lambda s:s.str.strip().ne("")).any(axis=1)].reset_index(drop=True)
    return out.fillna("")

df = build_frame(raw, HEADER_ROW, HEADER_DEPTH)
print(f"사용 헤더 행={HEADER_ROW}, 깊이={HEADER_DEPTH} | 추출 행={len(df)} / 컬럼={len(df.columns)}")
for i,c in enumerate(df.columns): print(f"{i:02d}: {c}")
display(df.head(3).T)

헤더 후보 (점수, 0부터 시작하는 행): [(29, 1), (10, 0), (6, 20), (6, 7), (5, 24), (5, 22), (5, 21), (5, 19)]
사용 헤더 행=1, 깊이=1 | 추출 행=30 / 컬럼=57
00: _excel_row
01: No
02: 접수번호
03: 하위작업
04: 접수일
05: 접수경로
06: 거래처구분
07: 거래처코드
08: 거래처명
09: 의뢰자명
10: 처리상태
11: 부서명
12: 처리담당자
13: 중요도
14: 모듈
15: 중분류
16: 프로그램명
17: 처리요청일자
18: 시스템유형
19: 업무유형
20: 문의유형
21: 요청내용
22: CMC담당자
23: 처리예정일자
24: 처리예정시각
25: 원인유형
26: 조치유형
27: 처리등급
28: 처리완료일
29: 예상 작업시간
30: 진행률
31: 대상 프로젝트
32: 작업시간
33: 조치 및 처리 의견
34: 프로그램 만족도
35: 처리구분
36: 빌드순번/CTS빌드
37: 파일명/PBL
38: 관련DB
39: 테스트케이스
40: 프로세스정의서
41: 기능분해도
42: 화면정의서
43: ERD
44: 테이블정의서
45: 인터페이스정의서
46: 승인여부
47: 승인자
48: 승인일자
49: 승인여부__2
50: 승인자__2
51: 승인일자__2
52: 배포경로 및 배포파일명
53: SVN버전
54: 배포일자
55: 배포승인일시
56: 배포승인시간


,0,1,2
_excel_row,3,4,5
No,30,29,28
접수번호,20260731001,20260729001,20260728004
하위작업,,,
접수일,20260731,20260729,20260728
접수경로,방문,방문,CALL
거래처구분,본사,본사,대외기업
거래처코드,00000,00000,11654
거래처명,(주)중외정보기술,(주)중외정보기술,(주)한국티이아이
의뢰자명,김경아,이준호,최정민


### 0-5. 의미 필드 매핑

In [6]:
ALIASES = {
 "ticket_id":["접수번호", "요구사항번호", "접수 no", "접수no"],
 "subtask":["하위작업", "세부작업", "작업순번"],
 "received_at":["접수일자", "접수일", "등록일자"],
 "status":["처리상태", "진행상태", "상태"],
 "system":["시스템", "시스템명", "구분", "서비스"],
 "category":["문의유형", "요청유형", "분류"],
 "title":["제목", "요구사항명", "요청제목", "문의제목"],
 "request":["요청내용", "요구사항", "문의내용", "접수내용", "요청사항", "내용"],
 "resolution":["조치내용", "조치의견", "처리내용", "처리결과", "해결내용", "완료내용"],
 "completed_at":["처리완료일", "완료일자", "완료일"],
 "work_hours":["작업시간", "소요시간", "투입시간"],
 "assignee":["처리담당자", "담당자", "담당"],
 "customer":["고객사명", "고객사", "고객"],
}
FIELD_OVERRIDE = {}  # 예: {"ticket_id":"접수정보 / 접수번호", "request":"문의내용"}
def flat(s): return re.sub(r"[^0-9a-zA-Z가-힣]", "", str(s).lower())
def match_col(key, choices):
    if key in FIELD_OVERRIDE:
        assert FIELD_OVERRIDE[key] in df.columns, f"열 없음: {FIELD_OVERRIDE[key]}"
        return FIELD_OVERRIDE[key]
    aliases = ALIASES[key]
    candidates = []
    for c in choices:
        cf = flat(c)
        for a in aliases:
            af = flat(a)
            if cf == af or cf.endswith(af) or af in cf:
                candidates.append((0 if cf == af else 1 if cf.endswith(af) else 2, len(cf), c)); break
    return sorted(candidates)[0][-1] if candidates else None
field_map = {k: match_col(k, df.columns) for k in ALIASES}
display(pd.DataFrame({"의미":list(field_map), "원본 컬럼":list(field_map.values())}))
assert field_map["ticket_id"] is not None, "접수번호 열을 지정하세요: FIELD_OVERRIDE['ticket_id']"
assert any(field_map[k] for k in ["request","title","resolution"]), "요청/조치 내용 컬럼을 수동 지정하세요."
print("전체 원본 행수:",len(df),"| 고유 접수번호:",df[field_map["ticket_id"]].astype(str).nunique())

,의미,원본 컬럼
0,ticket_id,접수번호
1,subtask,하위작업
2,received_at,접수일
3,status,처리상태
4,system,처리구분
5,category,문의유형
6,title,None
7,request,요청내용
8,resolution,None
9,completed_at,처리완료일


전체 원본 행수: 30 | 고유 접수번호: 27


### 0-6. 레코드 정규화

In [13]:
def clean_value(x):
    if pd.isna(x): return ""
    return re.sub(r"\s+", " ", str(x)).strip()

data = df.copy()
ticket_col = field_map["ticket_id"]
# 병합셀로 인해 접수번호가 아래 행에 공란으로 들어간 경우만 이어받음
raw_id = data[ticket_col].map(clean_value)
if raw_id.ne("").any(): data[ticket_col] = raw_id.replace("",pd.NA).ffill().fillna("")
records = []
for _,row in data.iterrows():
    record = {"_excel_row": int(row["_excel_row"])}
    for key,col in field_map.items(): record[key] = clean_value(row[col]) if col else ""
    record["ticket_id"] = re.sub(r"\.0$", "", record["ticket_id"])
    record["record_id"] = f"{SHEET_NAME}:{record['_excel_row']}"
    records.append(record)
records = [r for r in records if r['ticket_id'] or r['request'] or r['title']]
print("정규화 레코드:",len(records),"| 고유 접수번호:", len(set(r['ticket_id'] for r in records if r['ticket_id'])))
display(pd.DataFrame(records).drop(columns=["customer","assignee"],errors="ignore").head(5))

정규화 레코드: 30 | 고유 접수번호: 27


,_excel_row,ticket_id,subtask,received_at,status,system,category,title,request,resolution,completed_at,work_hours,record_id
0,3,20260731001,,20260731,배포승인,단순수정,프로그램 개선,,[프로그램] 평가 등록 화면 이동 로직 수정 요청 메인화면에서 출력되는 평개 갯수와...,,20260731,2,AS관리:3
1,4,20260729001,,20260729,처리완료,,데이터 수정,,[데이터] 발행 세금계산서 품목명 수정 요청 202607290002 20260729...,,20260729,0.5,AS관리:4
2,5,20260728004,,20260728,처리완료,,데이터 수정,,검사일자 수정,,20260729,0.5,AS관리:5
3,6,20260728003,,20260728,처리완료,,교육 진행,,병원용 라인어스 교육 진행 대상자 : 김범수 매니저,,20260724,0.5,AS관리:6
4,7,20260727006,,20260727,처리완료,,데이터 수정,,[데이터] 주문전표 삭제 요청 DO202607270003,,20260727,0.5,AS관리:7


### 0-7. 정형 검색

In [14]:
def structured_search(ticket_id=None, status=None, system=None, keyword=None, limit=30):
    hits = []
    for r in records:
        if ticket_id and r['ticket_id'] != str(ticket_id): continue
        if status and str(status) not in r['status']: continue
        if system and str(system).lower() not in r['system'].lower(): continue
        searchable = " ".join(r.get(k,"") for k in ['title','request','resolution','category','status','system'])
        if keyword and str(keyword).lower() not in searchable.lower(): continue
        hits.append(r)
    return hits[:limit]

def show_hits(hits):
    cols = ['record_id','ticket_id','status','system','title','request','resolution']
    display(pd.DataFrame([{k:r.get(k,"")[:160] for k in cols} for r in hits]))

example_id = next((r['ticket_id'] for r in records if r['ticket_id']),"")
print("실제 데이터의 첫 접수번호 검색:",example_id)
show_hits(structured_search(ticket_id=example_id))

실제 데이터의 첫 접수번호 검색: 20260731001


,record_id,ticket_id,status,system,title,request,resolution
0,AS관리:3,20260731001,배포승인,단순수정,,[프로그램] 평가 등록 화면 이동 로직 수정 요청 메인화면에서 출력되는 평개 갯수와...,


### 0-8. 정확한 그룹 집계

In [15]:
rframe = pd.DataFrame(records)
def group_metrics(group_key):
    if rframe.empty or not rframe[group_key].astype(str).str.strip().any():
        return pd.DataFrame(columns=[group_key,'행 수','고유 접수건'])
    sub = rframe.copy(); sub[group_key] = sub[group_key].replace("", "(미기재)")
    a = sub.groupby(group_key, dropna=False).agg(**{'행 수':('record_id','size'), '고유 접수건':('ticket_id',lambda x:x.replace('',pd.NA).nunique())})
    return a.sort_values('행 수',ascending=False).reset_index()
print("전체 행 수:",len(rframe),"| 고유 접수건:", rframe.ticket_id.replace('',pd.NA).nunique())
print("상태별 분포"); display(group_metrics('status'))
print("시스템별 분포"); display(group_metrics('system'))
if rframe.work_hours.astype(str).str.strip().any():
    nums = pd.to_numeric(rframe.work_hours.str.replace(',','',regex=False),errors='coerce')
    print('숫자로 해석 가능한 작업시간:',nums.notna().sum(),'건 / 합계:',nums.sum(),' (원본 시간 단위 확인 필수)')

전체 행 수: 30 | 고유 접수건: 27
상태별 분포


,status,행 수,고유 접수건
0,처리완료,22,22
1,배포승인,6,5
2,반려,1,1
3,처리중,1,1


시스템별 분포


,system,행 수,고유 접수건
0,(미기재),24,23
1,단순수정,5,4
2,형상변경,1,1


숫자로 해석 가능한 작업시간: 30 건 / 합계: 438.1  (원본 시간 단위 확인 필수)


### 0-9. LangChain 검색 문서 및 비식별화

In [16]:
from langchain_core.documents import Document

def redact(text):
    text = re.sub(r"[\w.+-]+@[\w.-]+\.[a-zA-Z]{2,}","[EMAIL]",str(text))
    text = re.sub(r"(?<!\d)01[016789][- ]?\d{3,4}[- ]?\d{4}(?!\d)","[PHONE]",text)
    text = re.sub(r"\b\d{6}[- ]?[1-4]\d{6}\b", "[PERSONAL_ID]",text)
    return text

def record_to_doc(r):
    body = '\n'.join([f"접수번호: {r['ticket_id']}",f"시스템: {r['system']}",f"유형: {r['category']}",f"상태: {r['status']}",f"제목: {r['title']}",f"요청내용: {r['request']}",f"조치내용: {r['resolution']}"])
    body = redact(body)
    return Document(page_content=body, metadata={"record_id":r['record_id'], "ticket_id":r['ticket_id'],"status":r['status'],"system":r['system'],"excel_row":r['_excel_row']})
documents = [record_to_doc(r) for r in records if r['request'] or r['title'] or r['resolution']]
print("문서 개수:",len(documents)); print(documents[0].page_content[:800] if documents else '문서 없음')

문서 개수: 30
접수번호: 20260731001
시스템: 단순수정
유형: 프로그램 개선
상태: 배포승인
제목: 
요청내용: [프로그램] 평가 등록 화면 이동 로직 수정 요청 메인화면에서 출력되는 평개 갯수와 메인화면에서 버튼클릭으로 들어왔을때 조회되는 리스트가 동일하게 수정 진행
조치내용: 


### 0-10. 선택적 Chroma 벡터 인덱스

In [17]:
vectorstore = None
if ENABLE_EXTERNAL_AI:
    assert os.environ.get("OPENAI_API_KEY"), "API 키가 필요합니다."
    from langchain_openai import OpenAIEmbeddings
    from langchain_chroma import Chroma
    if not documents: raise ValueError("인덱싱할 문서가 없습니다.")
    embeddings = OpenAIEmbeddings(model=EMBED_MODEL)
    vectorstore = Chroma(collection_name="mro_training_hybrid", embedding_function=embeddings)
    vectorstore.add_documents(documents=documents, ids=[d.metadata['record_id'] for d in documents])
    print("Chroma 인덱싱 완료:",len(documents))
else:
    print("안전 모드: API 임베딩 생성 생략 (ENABLE_EXTERNAL_AI=False)")

Chroma 인덱싱 완료: 30


### 0-11. Hybrid Retrieval 및 record_id 매핑

In [18]:
record_by_id = {r['record_id']:r for r in records}
def satisfies(r, filters):
    for k,v in (filters or {}).items():
        if v is None or v == '': continue
        if k == 'ticket_id' and r['ticket_id'] != str(v): return False
        if k in ['status','system'] and str(v).lower() not in r[k].lower(): return False
    return True

def hybrid_retrieve(question, filters=None, keyword=None, k=5):
    filters = filters or {}
    exact = [r for r in records if satisfies(r,filters) and (not keyword or keyword.lower() in ' '.join([r['title'],r['request'],r['resolution']]).lower())]
    # 모든 레코드를 무조건 결과로 만들지 않고, 조건이 없는 경우 키워드 매치만 정형 후보로 인정
    if not filters and not keyword: exact = []
    combined = {}
    for r in exact:
        combined[r['record_id']] = {"record":r,"score":2.0,"channels":["structured"]}
    if vectorstore is not None:
        for rank,doc in enumerate(vectorstore.similarity_search(question,k=min(12,max(k*3,8)))):
            rid = doc.metadata.get('record_id'); r=record_by_id.get(rid)
            if r is None or not satisfies(r,filters): continue
            if rid in combined:
                combined[rid]['score'] += 1.0/(rank+1)
                combined[rid]['channels'].append('semantic')
            else:
                combined[rid] = {'record':r,'score':1.0/(rank+1),'channels':['semantic']}
    return sorted(combined.values(), key=lambda h:-h['score'])[:k]



### 0-12. Custom Tool 공통 유틸리티

In [19]:
import json
from typing import Optional
from langchain_core.tools import tool

required_names = ["records", "rframe", "documents", "record_by_id", "structured_search", "group_metrics", "hybrid_retrieve", "record_to_doc", "redact"]
missing = [n for n in required_names if n not in globals()]
assert not missing, f"원본 Cell 1~16을 먼저 실행하세요. 없는 함수/변수: {missing}"

# 검색 결과로 내보낼 필드만 화이트리스트로 지정
PUBLIC_FIELDS = ("record_id", "ticket_id", "_excel_row", "status", "system", "category", "title", "request", "resolution")

def safe_record(r, max_chars=380):
    out = {}
    for key in PUBLIC_FIELDS:
        value = r.get(key, "")
        out["excel_row" if key == "_excel_row" else key] = (
            value if key == "_excel_row" else redact(str(value))[:max_chars]
        )
    return out

def dump_result(obj):
    return json.dumps(obj, ensure_ascii=False, default=str)

print(f"데이터 준비: {len(records)}개 행 레코드 / {len(documents)}개 검색 문서 / 외부 AI: {ENABLE_EXTERNAL_AI}")

데이터 준비: 30개 행 레코드 / 30개 검색 문서 / 외부 AI: True


### 0-13. 정확 검색 Tool

In [20]:
@tool
def search_mro_records(
    ticket_id: Optional[str] = None,
    status: Optional[str] = None,
    system: Optional[str] = None,
    keyword: Optional[str] = None,
    limit: int = 5,
) -> str:
    """MRO 요구사항 원본 Excel에서 접수번호 정확 검색 또는 상태·시스템·키워드 조건 검색. 행 근거를 반환합니다."""
    limit = max(1, min(int(limit), 20))
    # 정보 과다 노출 방지: 조건을 하나 이상 요구
    if not any([ticket_id, status, system, keyword]):
        return dump_result({"error": "최소 1개의 검색 조건을 입력하세요."})
    hits = structured_search(ticket_id=ticket_id, status=status, system=system, keyword=keyword, limit=limit)
    return dump_result({"count_returned": len(hits), "records": [safe_record(r) for r in hits]})

print(search_mro_records.name, search_mro_records.args_schema.model_json_schema()["properties"].keys())

search_mro_records dict_keys(['ticket_id', 'status', 'system', 'keyword', 'limit'])


### 0-14. 통계 Tool

In [21]:
@tool
def get_mro_statistics(group_by: str = "status", limit: int = 20) -> str:
    """MRO 원본 Excel의 처리상태(status), 시스템(system), 업무유형(category)별 행 수와 고유 접수건을 정확하게 집계합니다."""
    if group_by not in {"status", "system", "category"}:
        return dump_result({"error": "group_by는 status/system/category 중 하나여야 합니다."})
    frame = group_metrics(group_by)
    rows = frame.head(max(1, min(int(limit), 100))).to_dict(orient="records")
    for row in rows:
        row[group_by] = redact(str(row[group_by]))
        row["행 수"] = int(row["행 수"])
        row["고유 접수건"] = int(row["고유 접수건"])
    return dump_result({"group_by": group_by, "total_rows": int(len(rframe)),
                        "unique_tickets": int(rframe.ticket_id.replace('', pd.NA).nunique()),
                        "groups": rows, "truncated": len(frame) > len(rows)})

### 0-15. BM25/RRF 검색 Tool

In [22]:
from langchain_community.retrievers import BM25Retriever

# 기존 문서를 BM25 색인으로 사용. 인덱스는 한 번만 생성합니다.
BM25_INDEX = BM25Retriever.from_documents(documents) if documents else None
if BM25_INDEX is not None:
    BM25_INDEX.k = min(20, len(documents))

@tool
def search_similar_mro_cases(
    query: str,
    status: Optional[str] = None,
    system: Optional[str] = None,
    keyword: Optional[str] = None,
    top_k: int = 5,
) -> str:
    """MRO 유사사례 검색: BM25 키워드와 승인 시 Chroma 벡터를 RRF로 결합하고 상태/시스템 필터를 준수합니다."""
    top_k = max(1, min(int(top_k), 10))
    constraints = {"status": status, "system": system}
    def allowed(r):
        if status and status.lower() not in r["status"].lower(): return False
        if system and system.lower() not in r["system"].lower(): return False
        if keyword and keyword.lower() not in " ".join(r.get(k,"") for k in ("title","request","resolution")).lower(): return False
        return True

    scores, channels = {}, {}
    def merge_docs(docs, channel, candidate_rank_offset=0):
        for rank, doc in enumerate(docs, start=1):
            rid = doc.metadata.get("record_id")
            rec = record_by_id.get(rid)
            if rec is None or not allowed(rec):
                continue
            scores[rid] = scores.get(rid, 0.0) + 1/(60 + candidate_rank_offset + rank)
            channels.setdefault(rid, []).append(channel)

    if BM25_INDEX is not None:
        # 필터를 먼저 적용해야 후보 중 조건에 맞는 문서를 놓치지 않습니다.
        relevant_docs = [d for d in documents if allowed(record_by_id[d.metadata["record_id"]])]
        if relevant_docs:
            local_bm25 = BM25Retriever.from_documents(relevant_docs)
            local_bm25.k = min(len(relevant_docs), 40)
            merge_docs(local_bm25.invoke(query), "bm25")

    if vectorstore is not None:
        # 원본 Chroma 인덱스 재사용. 검색 후 제약 조건을 다시 확인합니다.
        merge_docs(vectorstore.similarity_search(query, k=min(len(documents), max(50, top_k * 10))), "chroma")

    ranking = sorted(scores, key=lambda rid: (-scores[rid], rid))[:top_k]
    return dump_result({"query": query, "retrieval_channels": ["bm25"] + (["chroma"] if vectorstore is not None else []),
                        "count_returned": len(ranking), "hits": [dict(safe_record(record_by_id[rid], 500),
                        rrf_score=round(scores[rid], 7), channels=channels[rid]) for rid in ranking]})

/tmp/ipykernel_445/1284465335.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


### 0-16. 상세 조회 Tool

In [23]:
@tool
def get_mro_record_detail(record_id: str) -> str:
    """검색 결과의 record_id(시트명:Excel 행 번호)로 MRO 요구사항의 원본 근거를 조회합니다."""
    r = record_by_id.get(record_id)
    if r is None:
        return dump_result({"found": False, "record_id": record_id, "message": "일치하는 Excel 행이 없습니다."})
    return dump_result({"found": True, "record": safe_record(r, max_chars=900)})

MRO_TOOLS = [search_mro_records, get_mro_statistics, search_similar_mro_cases, get_mro_record_detail]
print("등록한 Custom Tools:", [t.name for t in MRO_TOOLS])

등록한 Custom Tools: ['search_mro_records', 'get_mro_statistics', 'search_similar_mro_cases', 'get_mro_record_detail']


## 0-17. 보고서 생성 전 필수 검증

앞의 출력에서 `HEADER_ROW`, `HEADER_DEPTH`, `field_map`이 원본과 맞는지 반드시 확인하세요. 아래 검사는 데이터 및 Tool 연결을 확인합니다.

In [24]:
assert len(records)>0, "분석할 레코드가 없습니다. 헤더/필드 매핑을 확인하세요."
assert len(record_by_id)==len(records), "record_id 중복이 있습니다."
assert int(group_metrics('status')['행 수'].sum())==len(records) if not group_metrics('status').empty else True
assert search_similar_mro_cases.invoke({'query':'요구사항','top_k':2})
print('PASS: 원본 레코드, 그룹 통계, Hybrid Tool 준비 완료')

PASS: 원본 레코드, 그룹 통계, Hybrid Tool 준비 완료


---
# Part 4. Hybrid RAG 분석 결과 → MRO 결과 보고서 자동 생성

**학습목표:** 기존 `records`, `group_metrics`, `search_similar_mro_cases`, `get_mro_statistics`의 결과로 **수치가 검증되는 Word 보고서**, **근거 추적용 Excel 파일**, **감사 JSON**, **ZIP 묶음**을 생성합니다.

- **정확한 건수**: Pandas로 계산합니다. 행 수와 고유 접수건을 분리합니다.
- **관련 사례**: BM25(+Chroma) Tool로 수집하며 각 사례의 `record_id`/`ticket_id`/`excel_row`를 유지합니다.
- **문장 생성**: 선택 사항입니다. `ENABLE_EXTERNAL_AI=True`일 때만 ChatOpenAI를 사용하며, 기업 데이터 외부 전송 승인이 필요합니다.
- **보안**: 고객사/담당자 필드를 제외하며 정규식 마스킹을 적용합니다. 문서 원문 속 기업명·민감정보는 완전히 탐지되지 않을 수 있습니다. 검토 없이 외부 배포하지 마세요.

> 이 부분은 앞의 기존 노트북 셀 실행 후 진행합니다. 테스트 결과 수치는 실제 업로드 파일에서 계산되며, 임의의 샘플 KPI를 사실로 보고하지 않습니다.

## C-1. 보고서 패키지 설치 및 환경 확인

Excel 데이터가 로딩되어 있고 기존 Custom Tools가 등록되어 있는지 확인합니다.

In [25]:
%pip -q install -U python-docx openpyxl
from pathlib import Path
import datetime as dt
import json, re
import pandas as pd

need = ["records", "rframe", "group_metrics", "record_by_id", "redact", "get_mro_statistics", "search_similar_mro_cases"]
missing = [name for name in need if name not in globals()]
assert not missing, f"먼저 기존 노트북을 실행하세요: {missing}"
assert len(records)>0, "유효한 MRO 레코드가 없습니다. 원본 헤더/필드 매핑을 확인하세요."
REPORT_DIR = Path('/content/mro_reports' if Path('/content').exists() else '/mnt/data/mro_reports')
REPORT_DIR.mkdir(parents=True, exist_ok=True)
print('입력 파일:', Path(EXCEL_PATH).name, '| 레코드:',len(records), '| 출력:',REPORT_DIR)

입력 파일: 2026년_3분기_DO본부_MRO팀_접수일자기준-요구사항정의서(07월)_LineUs.xls | 레코드: 30 | 출력: /content/mro_reports


## C-2. 결과 보고서 질문 설정

보고서 분석 주제는 실제 Excel의 항목에 맞게 변경할 수 있습니다. 그룹 기준은 원본 정규화 필드인 `status`, `system`, `category`를 사용합니다.

In [26]:
REPORT_TITLE = '2026년 7월 MRO 요구사항 분석 결과 보고서'
REPORT_QUESTIONS = [
    'ERP 관련 MRO 요구사항과 처리 사례',
    '메일 자동 발송 관련 요구사항 및 처리 이력',
    '처리 지연이나 미완료 위험과 관련된 요구사항',
]
TOP_K_PER_QUESTION = 5
MAX_EVIDENCE_TOTAL = 15
USE_AI_NARRATIVE = False  # 사내 승인된 비식별 데이터에 한해 True로 변경
assert not USE_AI_NARRATIVE or ENABLE_EXTERNAL_AI, 'AI 요약 활성화에는 ENABLE_EXTERNAL_AI=True와 외부 전송 승인이 필요합니다.'
print('분석 주제:', REPORT_QUESTIONS)

분석 주제: ['ERP 관련 MRO 요구사항과 처리 사례', '메일 자동 발송 관련 요구사항 및 처리 이력', '처리 지연이나 미완료 위험과 관련된 요구사항']


## C-3. 검증 가능한 KPI와 그룹별 현황 생성

이 단계는 RAG가 아닌 **원본 레코드 기반의 결정적 집계**입니다. 모든 집계는 총 행 수와 비교해 정합성을 검증합니다.

In [27]:
def safe_string(value, limit=600):
    return redact(str(value or ''))[:limit]

def group_table(key):
    tbl = group_metrics(key).copy()
    if not tbl.empty:
        tbl[key] = tbl[key].fillna('(미기재)').map(lambda x: safe_string(x, 120))
        for c in ('행 수','고유 접수건'): tbl[c] = tbl[c].astype(int)
        assert int(tbl['행 수'].sum()) == len(records), f'{key} 그룹별 행 수 합산 불일치'
    return tbl

status_df = group_table('status')
system_df = group_table('system')
category_df = group_table('category')
unique_tickets = len({r['ticket_id'] for r in records if str(r['ticket_id']).strip()})
metrics = {
    '총 처리 이력 행 수': len(records),
    '고유 접수건': unique_tickets,
    '접수번호 미기재 행': sum(not str(r['ticket_id']).strip() for r in records),
    '원본 시트': SHEET_NAME,
    '보고서 작성일': dt.date.today().isoformat(),
}
print(metrics)
display(status_df.head(20),system_df.head(20),category_df.head(20))

{'총 처리 이력 행 수': 30, '고유 접수건': 27, '접수번호 미기재 행': 0, '원본 시트': 'AS관리', '보고서 작성일': '2026-11-18'}


,status,행 수,고유 접수건
0,처리완료,22,22
1,배포승인,6,5
2,반려,1,1
3,처리중,1,1


,system,행 수,고유 접수건
0,(미기재),24,23
1,단순수정,5,4
2,형상변경,1,1


,category,행 수,고유 접수건
0,데이터 수정,9,9
1,데이터 요청,5,5
2,프로그램 개선,5,5
3,프로그램 신규 개발,3,1
4,회의/출장,3,3
5,권한/예외 요청,2,2
6,응대(전화/메신저/이메일),2,2
7,교육 진행,1,1


## C-4. Custom Tool을 사용한 분석 근거 추출

각 질문에 대해 `search_similar_mro_cases.invoke()`를 실행하여 **record_id 단위로 중복 제거**합니다. 조회 결과는 반드시 `record_by_id`에서 다시 확인하고, 원본 행과 접수번호를 일치 검증합니다.

외부 AI가 비활성화되어 있으면 BM25 검색 기반으로 동작합니다.

In [28]:
def load_tool_json(tool_obj, payload):
    result = tool_obj.invoke(payload)
    if isinstance(result, str): return json.loads(result)
    if isinstance(result, dict): return result
    raise TypeError(f'지원하지 않는 Tool 결과 유형: {type(result)}')

question_evidence = []
for question in REPORT_QUESTIONS:
    payload = load_tool_json(search_similar_mro_cases, {'query': question,'top_k':TOP_K_PER_QUESTION})
    for hit in payload.get('hits', []):
        rid = str(hit.get('record_id',''))
        original = record_by_id.get(rid)
        if not original: continue
        assert str(hit.get('ticket_id','')) == str(original['ticket_id']), '접수번호 근거 불일치'
        assert str(hit.get('excel_row',hit.get('_excel_row',''))) == str(original['_excel_row']), '원본 Excel 행 근거 불일치'
        question_evidence.append({
            '질문':question,'record_id':rid,'접수번호':safe_string(original['ticket_id'],120),
            'Excel 행':int(original['_excel_row']),
            '상태':safe_string(original.get('status'),120),'시스템':safe_string(original.get('system'),120),
            '제목':safe_string(original.get('title'),250),
            '요청내용':safe_string(original.get('request'),550),
            '조치내용':safe_string(original.get('resolution'),550),
            '검색채널':','.join(hit.get('channels',[])),
            'RRF 점수':hit.get('rrf_score',None),
        })

seen=set(); evidence=[]
for item in question_evidence:
    if item['record_id'] not in seen:
        evidence.append(item);seen.add(item['record_id'])
evidence=evidence[:MAX_EVIDENCE_TOTAL]
evidence_df=pd.DataFrame(evidence)
print('유효 근거:',len(evidence),'개 | 중복 제거 완료')
if not evidence_df.empty: display(evidence_df[['질문','접수번호','Excel 행','제목','검색채널']])
else: print('검색된 근거가 없어 사례 분석을 생략합니다. 집계는 계속 유효합니다.')

유효 근거: 12 개 | 중복 제거 완료


,질문,접수번호,Excel 행,제목,검색채널
0,ERP 관련 MRO 요구사항과 처리 사례,20260707004,24,,"bm25,chroma"
1,ERP 관련 MRO 요구사항과 처리 사례,20260707004,22,,"bm25,chroma"
2,ERP 관련 MRO 요구사항과 처리 사례,20260707004,23,,"bm25,chroma"
3,ERP 관련 MRO 요구사항과 처리 사례,20260708001,21,,"bm25,chroma"
4,ERP 관련 MRO 요구사항과 처리 사례,20260708002,20,,"bm25,chroma"
5,메일 자동 발송 관련 요구사항 및 처리 이력,20260722002,11,,"bm25,chroma"
6,메일 자동 발송 관련 요구사항 및 처리 이력,20260720002,12,,"bm25,chroma"
7,메일 자동 발송 관련 요구사항 및 처리 이력,20260708004,18,,"bm25,chroma"
8,처리 지연이나 미완료 위험과 관련된 요구사항,20260703001,29,,"bm25,chroma"
9,처리 지연이나 미완료 위험과 관련된 요구사항,20260702004,30,,"bm25,chroma"


## C-5. 보고서 해설 생성 (기본값: 비 AI / 선택적 생성)

AI를 사용하지 않는 기본 모드에서는 수치 중심의 검증 가능한 해설을 작성합니다. AI 모드에서도 **원본 KPI 수치를 변경하지 못하게** 별도 섹션에만 텍스트를 사용합니다.

In [29]:
def deterministic_narrative(metrics, statuses, evidence):
    lines = [f"원본 {metrics['원본 시트']} 시트를 기준으로 처리 이력 {metrics['총 처리 이력 행 수']}행, 고유 접수번호 {metrics['고유 접수건']}건을 확인했습니다."]
    if not statuses.empty:
        top=statuses.iloc[0]
        lines.append(f"원본에 기재된 상태값 중 행 수가 가장 많은 값은 '{top['status']}'이며 해당 {int(top['행 수'])}행입니다. 이는 업무 완료율을 뜻하지 않습니다.")
    lines.append(f"검색 근거로 확인된 유사사례는 {len(evidence)}개입니다. 사례는 검색 결과의 일부이며 전체 모집단의 대표 표본은 아닙니다.")
    lines.append('작업시간의 단위, 완료 상태의 정의 및 접수번호 중복의 업무 의미는 현업 담당자 확인 후 성과지표로 해석해야 합니다.')
    return '\n'.join(lines)

narrative=deterministic_narrative(metrics,status_df,evidence)
if USE_AI_NARRATIVE:
    from langchain_openai import ChatOpenAI
    from langchain_core.messages import SystemMessage, HumanMessage
    approved_context={'metrics':metrics,'status':status_df.head(10).to_dict('records'),'evidence':evidence[:8]}
    messages=[
        SystemMessage(content='당신은 기업 MRO 보고서 작성 보조자입니다. 주어진 수치와 근거만 사용하십시오. 수치를 새로 계산하거나 원인을 단정하지 마십시오. 개선 제안은 [제안]으로 표기하고, 출처 없는 결론을 금지합니다. 한국어 3~5문장으로 작성하세요.'),
        HumanMessage(content=json.dumps(approved_context,ensure_ascii=False,default=str)),
    ]
    narrative=ChatOpenAI(model=CHAT_MODEL,temperature=0).invoke(messages).content
print(narrative)

원본 AS관리 시트를 기준으로 처리 이력 30행, 고유 접수번호 27건을 확인했습니다.
원본에 기재된 상태값 중 행 수가 가장 많은 값은 '처리완료'이며 해당 22행입니다. 이는 업무 완료율을 뜻하지 않습니다.
검색 근거로 확인된 유사사례는 12개입니다. 사례는 검색 결과의 일부이며 전체 모집단의 대표 표본은 아닙니다.
작업시간의 단위, 완료 상태의 정의 및 접수번호 중복의 업무 의미는 현업 담당자 확인 후 성과지표로 해석해야 합니다.


## C-6. Word 보고서(.docx) 생성

보고서에는 분석 범위, KPI, 상태별/시스템별 현황, 검색 근거, 개선 검토 항목, 데이터 한계를 포함합니다.

In [30]:
from docx import Document as WordDocument
from docx.shared import Pt, Inches, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT

def add_word_table(doc, frame, max_rows=20):
    if frame is None or frame.empty:
        doc.add_paragraph('표시 가능한 데이터가 없습니다.');return
    table=doc.add_table(rows=1, cols=len(frame.columns))
    table.style='Light Shading Accent 1'
    for j, name in enumerate(frame.columns): table.rows[0].cells[j].text=str(name)
    for _,row in frame.head(max_rows).iterrows():
        cells=table.add_row().cells
        for j,value in enumerate(row): cells[j].text=str(value)[:350]

def create_report_docx():
    doc=WordDocument()
    sec=doc.sections[0];sec.top_margin=Inches(0.75);sec.bottom_margin=Inches(0.75)
    normal=doc.styles['Normal'];normal.font.name='Malgun Gothic';normal.font.size=Pt(10)
    heading=doc.add_heading(REPORT_TITLE,0);heading.alignment=WD_ALIGN_PARAGRAPH.CENTER
    doc.add_paragraph(f"작성일: {metrics['보고서 작성일']} | 대상 시트: {metrics['원본 시트']} | 출처: {Path(EXCEL_PATH).name}")
    doc.add_heading('1. 분석 범위 및 목적',1)
    doc.add_paragraph('MRO 요구사항 원본 Excel의 정규화 결과를 대상으로 정형 통계와 Hybrid RAG 검색 사례를 결합하여 현황 및 후속 검토 사항을 요약합니다.')
    doc.add_heading('2. 주요 KPI',1)
    add_word_table(doc,pd.DataFrame([{'항목':k,'값':v} for k,v in metrics.items()]))
    doc.add_heading('3. 핵심 분석 결과',1);doc.add_paragraph(narrative)
    doc.add_heading('4. 처리상태별 현황',1);add_word_table(doc,status_df)
    doc.add_heading('5. 시스템별 현황',1);add_word_table(doc,system_df)
    doc.add_heading('6. 업무유형별 현황',1);add_word_table(doc,category_df)
    doc.add_heading('7. Hybrid RAG 검색 근거',1)
    if not evidence:
        doc.add_paragraph('조건에 해당하는 확인 가능한 검색 근거가 없습니다.')
    for i,item in enumerate(evidence,1):
        doc.add_heading(f"사례 {i}. {item['제목'] or '(제목 없음)'}",2)
        doc.add_paragraph(f"질문: {item['질문']} | 접수번호: {item['접수번호']} | Excel 행: {item['Excel 행']} | record_id: {item['record_id']}")
        doc.add_paragraph('요청: '+(item['요청내용'] or '(기재 없음)'))
        doc.add_paragraph('조치: '+(item['조치내용'] or '(기재 없음)'))
    doc.add_heading('8. 개선 검토 및 후속 조치',1)
    for t in ['반복 요구사항은 담당자가 원본을 확인한 후 업무 표준화 후보로 검토',
              '상태별 건수와 고유 접수건을 분리해 지표 정의 명확화',
              '작업시간 단위·완료 기준·부서별 역할을 확정한 뒤 리드타임 KPI 도입 검토']:
        doc.add_paragraph('검토 제안: '+t,style='List Bullet')
    doc.add_heading('9. 데이터 품질 및 해석상 한계',1)
    doc.add_paragraph('다층 헤더 자동 탐지와 필드 매핑이 올바른지 원본 시트에서 검수해야 합니다. 검색 사례는 전체 건수의 무작위 표본이 아닙니다. 검색되지 않은 사실을 부재로 단정할 수 없습니다.')
    doc.add_paragraph('본 결과물은 정규식 기반 일부 마스킹만 적용되며 기업/개인 식별정보가 잔존할 수 있습니다. 외부 공유 이전 담당자 승인과 보안 검토가 필요합니다.')
    out=REPORT_DIR/'MRO_Hybrid_RAG_Analysis_Report.docx';doc.save(out)
    return out

word_file=create_report_docx()
print('Word 보고서:',word_file)

Word 보고서: /content/mro_reports/MRO_Hybrid_RAG_Analysis_Report.docx


## C-7. Excel 근거자료(.xlsx) 및 감사 기록(.json)

Excel 파일에는 KPI, 상태별 통계, 시스템별 통계, 업무유형별 통계, 원본 근거 추적 시트가 만들어집니다.

In [31]:
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

excel_file=REPORT_DIR/'MRO_Hybrid_RAG_Report_Evidence.xlsx'
with pd.ExcelWriter(excel_file,engine='openpyxl') as writer:
    pd.DataFrame([{'항목':k,'값':v} for k,v in metrics.items()]).to_excel(writer,sheet_name='KPI',index=False)
    status_df.to_excel(writer,sheet_name='상태별 현황',index=False)
    system_df.to_excel(writer,sheet_name='시스템별 현황',index=False)
    category_df.to_excel(writer,sheet_name='업무유형별 현황',index=False)
    evidence_df.to_excel(writer,sheet_name='검색 근거',index=False) if not evidence_df.empty else pd.DataFrame(columns=['검색 결과 없음']).to_excel(writer,sheet_name='검색 근거',index=False)
    for sheet in writer.book:
        sheet.freeze_panes='A2';sheet.auto_filter.ref=sheet.dimensions
        for cell in sheet[1]:
            cell.font=Font(bold=True,color='FFFFFF')
            cell.fill=PatternFill('solid',fgColor='17365D')
            cell.alignment=Alignment(wrap_text=True)
        for col in sheet.columns:
            col_letter=get_column_letter(col[0].column)
            max_len=max((len(str(cell.value or '')) for cell in list(col)[:120]),default=5)
            sheet.column_dimensions[col_letter].width=min(55,max(14,max_len+2))
        sheet.row_dimensions[1].height=28

audit_file=REPORT_DIR/'MRO_Hybrid_RAG_Audit.json'
audit={
    'source_excel':Path(EXCEL_PATH).name,'source_sheet':SHEET_NAME,
    'report_date':metrics['보고서 작성일'],'metrics':metrics,
    'retrieval_questions':REPORT_QUESTIONS,'unique_evidence':evidence,
    'validation':{'groups_sum_match':all(int(f['행 수'].sum())==len(records) for f in (status_df,system_df,category_df) if not f.empty),
                  'unique_record_ids':len({x['record_id'] for x in evidence})==len(evidence)},
    'limitations':['RAG 검색 사례는 전체 모집단의 대표 표본이 아님','원본 필드 매핑 및 개인/기업정보 확인 필요']
}
audit_file.write_text(json.dumps(audit,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
assert audit['validation']['groups_sum_match'] and audit['validation']['unique_record_ids']
print('Excel:',excel_file,'| JSON:',audit_file)

Excel: /content/mro_reports/MRO_Hybrid_RAG_Report_Evidence.xlsx | JSON: /content/mro_reports/MRO_Hybrid_RAG_Audit.json


## C-8. 통합 PASS/FAIL 검증 및 다운로드

실제 데이터 수치는 재계산하며, Word/Excel/JSON 파일 존재 여부와 출처 추적 일치 여부를 확인합니다.

In [32]:
from zipfile import ZipFile
assert word_file.exists() and word_file.stat().st_size > 1000
assert excel_file.exists() and excel_file.stat().st_size > 1000
assert audit_file.exists() and audit_file.stat().st_size > 100
assert unique_tickets == int(rframe.ticket_id.replace('',pd.NA).nunique())
assert all(x['record_id'] in record_by_id for x in evidence)
zip_file=REPORT_DIR/'MRO_Hybrid_RAG_Report_Package.zip'
with ZipFile(zip_file,'w') as z:
    for p in [word_file,excel_file,audit_file]:z.write(p,arcname=p.name)
print('PASS: 집계·출처·파일 검증 완료')
print('패키지:',zip_file)
try:
    from google.colab import files
    files.download(str(zip_file))
except ImportError:
    print('Colab이 아닌 환경에서는 위 경로의 ZIP 파일을 직접 사용하세요.')

PASS: 집계·출처·파일 검증 완료
패키지: /content/mro_reports/MRO_Hybrid_RAG_Report_Package.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 학습 정리 및 평가 기준

- **정형 집계:** KPI와 상태/시스템/업무유형 행 수 합계가 원본과 일치하는가?
- **검색 근거:** 각 사례에 접수번호, `record_id`, 원본 Excel 행 번호가 연결되는가?
- **생성형 AI 통제:** 원본에 없는 완료율/절감 효과/원인을 임의 생성하지 않았는가?
- **보안:** 외부 AI 사용 전 승인, 비식별화, 결과물 검토가 이루어졌는가?
- **보고서 완성도:** Word 결과 보고서, Excel 근거표, 감사 JSON이 함께 생성되는가?

**심화 과제:** `Deep Agent`가 보고서 생성 여부를 제안하도록 하되, 파일 내보내기는 사용자가 확인한 후 별도 승인된 워크플로에서 수행하도록 개선합니다.